# Quantum Noise Mitigation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThisIsBarney/quantx/blob/master/notebooks/09_noise_mitigation.ipynb)  [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ThisIsBarney/quantx/master?filepath=notebooks/09_noise_mitigation.ipynb)

This notebook demonstrates classical post-processing techniques to reduce noise in quantum computations:

1. **Zero-Noise Extrapolation (ZNE)** — extrapolate to zero noise by running at multiple noise levels
2. **Richardson Extrapolation** — polynomial extrapolation for smooth convergence
3. **Probabilistic Error Cancellation (PEC)** — quasi-probability decomposition
4. **Readout Error Mitigation** — calibration matrix inversion

All examples run in pure Python (no C++ library required).

In [ ]:
# Install quantx-engine (needed in Colab / Binder)
try:
    import quantx_engine  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           'quantx-engine'])
    import quantx_engine  # noqa: F401
print('quantx_engine version:', quantx_engine.__version__)

In [ ]:
import numpy as np
import quantx_engine
print('quantx_engine version:', quantx_engine.__version__)

## 1. Zero-Noise Extrapolation (ZNE)

ZNE amplifies the noise by scale factors $\lambda_1 < \lambda_2 < \ldots$ (e.g., gate folding),
measures expectation values $E(\lambda_i)$, then fits a model and extrapolates to $\lambda=0$.

### Setting up an executor

We build a simple noisy statevector executor using `make_sv_executor` and depolarizing noise.

In [ ]:
from quantx_engine.mitigation import zne_mitigate, make_sv_executor, RichardsonExtrapolator
from quantx_engine.gpu import GpuStatevector

# Bell circuit: |Φ+⟩ = (|00⟩+|11⟩)/√2
# For this state, ⟨Z⊗Z⟩ = 1.0  (the ideal expectation value we want to recover)
bell_ops = [
    {"name": "H",  "qubits": [0], "params": []},
    {"name": "CX", "qubits": [0, 1], "params": []},
]

# Observable: ⟨Z⊗Z⟩ = P(00) + P(11) - P(01) - P(10)
def zz_observable(sv: GpuStatevector) -> float:
    p = sv.probabilities()  # shape (4,) for 2-qubit state
    return float(p[0] + p[3] - p[1] - p[2])

# Noisy executor: runs the circuit, then applies depolarizing noise.
# Gate folding (scale factor > 1) increases the number of gates,
# which increases the total accumulated noise — exactly what ZNE requires.
p_noise = 0.04  # 4% depolarizing per gate

def noisy_executor(ops: list) -> float:
    sv = GpuStatevector(n_qubits=2, backend="numpy")
    for op in ops:
        sv.apply_gate(op["name"], op.get("qubits", []), op.get("params", []))
    ideal = zz_observable(sv)
    # Depolarizing noise on a 2-qubit observable decays as (1-4p/3)^n ≈ (1-2p)^n
    n_gates = len(ops)
    return ideal * (1 - 2 * p_noise) ** n_gates

# Noiseless value (ideal)
noiseless_executor = make_sv_executor(2, zz_observable)
ideal_val = noiseless_executor(bell_ops)

# Noisy value at scale 1
noisy_val = noisy_executor(bell_ops)

print(f'Bell circuit ⟨Z⊗Z⟩:')
print(f'  Ideal (noiseless): {ideal_val:.6f}')
print(f'  Noisy (scale=1):   {noisy_val:.6f}  (error: {abs(noisy_val - ideal_val):.4f})')

In [ ]:
# ZNE with Richardson extrapolation
# zne_mitigate(ops, executor, scale_factors, extrapolator) is the correct call signature
scale_factors = [1.0, 3.0, 5.0]

result_rich = zne_mitigate(
    bell_ops,
    noisy_executor,
    scale_factors=scale_factors,
    extrapolator=RichardsonExtrapolator(),
)

print('ZNE (Richardson extrapolation):')
print(f'  Scale factors:  {result_rich.scale_factors}')
print(f'  Noisy values:   {[f"{v:.4f}" for v in result_rich.noisy_values]}')
print(f'  Mitigated ⟨ZZ⟩: {result_rich.mitigated_value:.6f}  (ideal: {ideal_val:.6f})')
print(f'  Error: noisy={abs(noisy_val - ideal_val):.4f} → mitigated={abs(result_rich.mitigated_value - ideal_val):.4f}')

In [ ]:
from quantx_engine.mitigation import PolyExtrapolator, ExpExtrapolator

# Compare all three extrapolators
results = {}
for name, ext in [
    ('Richardson', RichardsonExtrapolator()),
    ('Polynomial', PolyExtrapolator()),
    ('Exponential', ExpExtrapolator()),
]:
    r = zne_mitigate(bell_ops, noisy_executor, scale_factors=scale_factors, extrapolator=ext)
    results[name] = r.mitigated_value

print(f'ZNE comparison (ideal = {ideal_val:.4f}, noisy = {noisy_val:.4f}):')
print(f'{"Method":>12}  {"Mitigated":>12}  {"Error":>8}')
print('-' * 38)
for name, val in results.items():
    print(f'{name:>12}  {val:>12.6f}  {abs(val - ideal_val):>8.4f}')

## 2. Richardson Extrapolation

`RichardsonExtrapolation` fits the noise-scaled expectation values
$E(\lambda_i)$ with a polynomial and extrapolates to $\lambda \to 0$.

It is equivalent to the Richardson polynomial limit and achieves $(n-1)$-th order
noise cancellation with $n$ scale factors.

In [ ]:
from quantx_engine.mitigation import RichardsonExtrapolation

# Synthetic data: E(λ) = 1.0 - 0.1*λ - 0.02*λ² (model a noisy expectation value)
def true_func(lam):
    return 1.0 - 0.1 * lam - 0.02 * lam**2

scale_factors = [1.0, 2.0, 3.0, 4.0]
values = [true_func(s) for s in scale_factors]

extrapolator = RichardsonExtrapolation(scale_factors)
mitigated = extrapolator.extrapolate(values)

true_zero = true_func(0.0)  # = 1.0

print('Richardson Extrapolation example:')
print(f'  Scale factors: {scale_factors}')
print(f'  Values:        {[f"{v:.4f}" for v in values]}')
print(f'  Extrapolated:  {mitigated:.8f}')
print(f'  True zero:     {true_zero:.8f}')
print(f'  Error:         {abs(mitigated - true_zero):.2e}')

assert abs(mitigated - true_zero) < 1e-8, f'Richardson extrapolation error too large: {abs(mitigated - true_zero)}'
print('Accuracy check passed  ✓')

In [ ]:
# Convergence: more scale factors → better accuracy for polynomial noise models
print('Richardson convergence with increasing number of scale factors:')
print(f'{"n_scales":>10}  {"extrapolated":>14}  {"error":>12}')
print('-' * 42)

for n in [2, 3, 4, 5]:
    sf = list(range(1, n + 1))
    vals = [true_func(s) for s in sf]
    ext = RichardsonExtrapolation(sf).extrapolate(vals)
    err = abs(ext - 1.0)
    print(f'{n:>10}  {ext:>14.8f}  {err:>12.2e}')

## 3. Probabilistic Error Cancellation (PEC)

PEC represents the ideal gate as a quasi-probability distribution over implementable noisy operations.
We use the `QuasiProbabilityDecomposition` for a depolarizing noise model.

In [ ]:
from quantx_engine.mitigation import QuasiProbabilityDecomposition

# Quasi-probability decomposition for depolarizing noise p=0.02
p = 0.02
qpd = QuasiProbabilityDecomposition.from_depolarizing(p)   # single-qubit channel

print('Quasi-probability decomposition for depolarizing(p=0.02):')
print(f'  1-norm γ (sampling overhead): {qpd.gamma:.4f}')
print(f'  Sampling overhead (γ²):       {qpd.gamma**2:.4f}x')
print(f'  Number of Pauli channels:     {len(qpd.basis)}')
# basis holds (Pauli, sign, |q_i|/γ), so q_i = sign · (|q_i|/γ) · γ
for label, sign, sample_prob in qpd.basis:
    print(f'    {label}: {sign * sample_prob * qpd.gamma:+.6f}')

In [ ]:
from quantx_engine.mitigation import pec_mitigate

# pec_mitigate(ops, noisy_executor, noise_prob, n_samples, seed) — correct signature
pec_result = pec_mitigate(
    bell_ops,
    noisy_executor,
    noise_prob=p_noise,
    n_samples=300,
    seed=42,
)

print('PEC mitigation of Bell ⟨Z⊗Z⟩:')
print(f'  Ideal value:       {ideal_val:.6f}')
print(f'  Noisy (scale=1):   {noisy_val:.6f}')
print(f'  PEC mitigated:     {pec_result.mitigated_value:.6f}  (std err: {pec_result.std_error:.4f})')
print(f'  Circuit 1-norm γ:  {pec_result.gamma:.4f}  (sampling overhead γ² = {pec_result.gamma**2:.4f}x)')

# PEC is unbiased only for the noise its QPD inverts.  This toy executor damps ⟨ZZ⟩
# per gate instead, and pec_mitigate leaves the CX untwirled, so the estimator's mean
# (≈0.850 over 10⁵ samples) barely moves off the unmitigated 0.846.  At 300 samples
# the difference printed below is sampling noise: other seeds land below 0.846.
print(f'\n  Error: noisy={abs(noisy_val - ideal_val):.4f} → PEC={abs(pec_result.mitigated_value - ideal_val):.4f}')

## 4. Noise Model Library

QuantX includes a comprehensive noise model library. Here we demonstrate
building composite noise models and applying a channel to a density matrix.

In [ ]:
from quantx_engine.noise import (
    DepolarizingNoise,
    ThermalRelaxationNoise,
    CompositeNoise,   # CompositeNoise combines multiple NoiseModel instances
)

# Build a composite noise model.
# NoiseModel is an abstract base class — use CompositeNoise to combine channels.
# The native backend runs one `noise` kind per circuit (plus depolarizing_prob), so
# CompositeNoise takes at most one noise-kind model: adding ReadoutNoise here raises
# NoiseModelError instead of silently dropping the T1 channel.
model = CompositeNoise([
    DepolarizingNoise(prob=0.005),           # 0.5% depolarizing
    ThermalRelaxationNoise(
        T1=100_000.0,   # 100 µs T1 relaxation time (all times in ns)
        T2=60_000.0,    # 60 µs T2 dephasing time
        t_gate=50.0,    # 50 ns gate time
    ),
])

model_dict = model.to_dict()

print('Composite noise model (2 channels combined):')
for key, val in sorted(model_dict.items()):
    print(f'  {key}: {val}')

print(f'\nTotal keys in model dict: {len(model_dict)}')

In [ ]:
# Apply the amplitude-damping channel to a density matrix
from quantx_engine.noise import AmplitudeDampingChannel

gamma = 0.1  # 10% decay probability
adc = AmplitudeDampingChannel(gamma=gamma)

# |1⟩ decays to |0⟩ with probability γ
rho_1 = np.array([[0, 0], [0, 1]], dtype=complex)   # |1⟩⟨1|
out = adc.apply(rho_1)
print(f'Amplitude damping (γ={gamma}) applied to |1⟩⟨1|:')
print(out.real)
assert np.allclose(np.diag(out).real, [gamma, 1 - gamma], atol=1e-12)

# The channel is trace-preserving: check on a random mixed state
rng = np.random.default_rng(0)
A = rng.normal(size=(2, 2)) + 1j * rng.normal(size=(2, 2))
rho = A @ A.conj().T
rho /= np.trace(rho)
tr = np.trace(adc.apply(rho)).real
print(f'\nTrace after the channel on a random state: {tr:.12f}')
assert abs(tr - 1.0) < 1e-12, 'Trace not preserved!'
print('Population and trace checks passed  ✓')

## 5. Noise Characterization via Randomized Benchmarking

The `StandardRB` class runs a randomized benchmarking experiment to estimate
the average gate error per Clifford (EPC).

In [ ]:
from quantx_engine.rb_qv import StandardRB

# Simulated depolarizing noise at 1% per Clifford
p_gate = 0.01

# Run RB with sequences of lengths [1, 2, 4, 8, 16, 32]
rb = StandardRB(n_qubits=1, p_error=p_gate, seed=42)
rb_result = rb.run(sequence_lengths=[1, 2, 4, 8, 16, 32], n_sequences=20)

# EPC from fit
epc = rb_result.epc
print(f'Randomized Benchmarking:')
print(f'  Injected depol. p:   {p_gate:.4f}')
print(f'  Expected EPC (p/2):  {p_gate / 2:.4f}')
print(f'  Fitted EPC:          {epc:.4f}')
print(f'  A (amplitude):       {rb_result.A:.4f}')
print(f'  p (decay rate):      {rb_result.p_decay:.4f}')

# EPC = (d−1)/d · p for a depolarizing channel of strength p; d = 2 for one qubit
assert abs(epc - p_gate / 2) < 1e-3, f'RB EPC {epc:.4f} far from expected {p_gate / 2:.4f}'
print('EPC consistency check passed  ✓')

## Summary

| Technique | Key benefit | QuantX module |
|-----------|------------|---------------|
| ZNE (polynomial) | Simple, no model of noise needed | `mitigation.zne_mitigate` |
| ZNE (Richardson) | Exact for polynomial noise, order-$(n-1)$ cancellation | `mitigation.RichardsonExtrapolation` |
| PEC | Unbiased, treats arbitrary noise | `mitigation.pec_mitigate` |
| Noise model library | Build composite noise for simulation | `noise.NoiseModel` |
| Randomized benchmarking | Measure average gate fidelity in hardware | `rb_qv.StandardRB` |

**When to use which technique:**
- **ZNE**: Low overhead, great for near-term devices; polynomial extrapolation works well for smooth noise
- **Richardson**: Best when noise scales polynomially with gate count
- **PEC**: Optimal for known noise models; overhead grows as $(1+2p)^{2n}$
- **RB**: Characterize your device before choosing a mitigation strategy

Next: **[01 — Getting Started](01_getting_started.ipynb)**